In [ ]:
%matplotlib widget

import ipywidgets as widgets
from ipywidgets import HBox, VBox, Box, Layout, Output, Label
from IPython.display import display, Latex, Markdown, Image, Math

import params as st
from model import nonlinSys, stationarySys
from feedforward import trajectory, feedForwardIO, feedForwardFlat

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

In [ ]:
def make_box_layout():
     return widgets.Layout(
        border='solid 1px black',
        margin='0px 5px 5px 0px',
        padding='2px 2px 2px 2px'
     )

# Steuerung des Zweitanksystems: Eingangs-Ausgangslinearisierung und flacher Ausgang

In [ ]:
imag = Image("../images/twoTank.png", width=450)

outL = Output()
outR = Output()

with outL:
    display(imag)
with outR:
    display(Markdown(r"""
**Modellgleichungen**

Zufluss $u(t) \geq 0$ der Pumpe in Tank 1, Ausfluss nach Torricelli $q_i(h_i) = a_i \sqrt{2 g h_i}$
\begin{align*}
    A_1 \dot{h}_1(t) & = u(t) - q_1(h_1(t)) \\
    A_2 \dot{h}_2(t) & = q_1(h_1(t)) - q_2(h_2(t))
\end{align*}

Verglichen werden zwei Ausgänge für denselben Arbeitspunktwechsel
- $y = h_1$: relativer Grad $r = 1 < n$, die Steuerung folgt aus der Eingangs-Ausgangslinearisierung, $h_2$ stellt sich über die interne Dynamik ein
- $y = h_2$: relativer Grad $r = 2 = n$, $h_2$ ist ein flacher Ausgang, beide Füllstände und $u$ sind durch $y$ und dessen Ableitungen festgelegt
    """))
cols = HBox([outL, outR], layout=Layout(display='flex', flex_flow='row', justify_content='space-around', align_items='center'))
display(cols)

## Relativer Grad

Eingangsaffine Darstellung $\dot{\boldsymbol{x}} = \boldsymbol{f}(\boldsymbol{x}) + \boldsymbol{g}(\boldsymbol{x}) u$ mit $\boldsymbol{x} = (h_1, h_2)^\intercal$

In [ ]:
h1, h2, u = sp.symbols('h_1 h_2 u', positive=True)
A1, A2, a1, a2, g = sp.symbols('A_1 A_2 a_1 a_2 g', positive=True)

x = sp.Matrix([h1, h2])
f = sp.Matrix([-a1 * sp.sqrt(2 * g * h1) / A1,
               (a1 * sp.sqrt(2 * g * h1) - a2 * sp.sqrt(2 * g * h2)) / A2])
gv = sp.Matrix([1 / A1, 0])

def lie(fun, vec):
    return sp.simplify((sp.Matrix([fun]).jacobian(x) * vec)[0])

display(Math(r"\boldsymbol{f}(\boldsymbol{x}) = " + sp.latex(f) + r",\quad \boldsymbol{g}(\boldsymbol{x}) = " + sp.latex(gv)))

In [ ]:
display(Math(r"y = h_1:\quad L_{\boldsymbol{g}} h_1 = " + sp.latex(lie(h1, gv)) + r" \neq 0 \quad\Rightarrow\quad r = 1"))
Lfh2 = lie(h2, f)
display(Math(r"y = h_2:\quad L_{\boldsymbol{g}} h_2 = " + sp.latex(lie(h2, gv)) + r",\quad L_{\boldsymbol{g}} L_{\boldsymbol{f}} h_2 = " + sp.latex(lie(Lfh2, gv)) + r" \neq 0 \quad\Rightarrow\quad r = 2 = n"))

## Ausgang $y = h_1$: Byrnes-Isidori-Normalform und Nulldynamik

Mit $\xi = h_1$ und $\eta = h_2$ (wegen $L_{\boldsymbol{g}} \eta = 0$) lautet die Normalform
\begin{align*}
    \dot{\xi} & = \frac{1}{A_1}\left(u - a_1 \sqrt{2 g \xi}\right), &
    \dot{\eta} & = \frac{1}{A_2}\left(a_1 \sqrt{2 g \xi} - a_2 \sqrt{2 g \eta}\right).
\end{align*}
Für $y(t) \equiv y^*$ ergibt sich die Nulldynamik

In [ ]:
yS, eta = sp.symbols('y^* eta', positive=True)
zeroDyn = f[1].subs({h1: yS, h2: eta})
etaS = sp.solve(zeroDyn, eta)[0]
lam = sp.simplify(sp.diff(zeroDyn, eta).subs(eta, etaS))

display(Math(r"\dot{\eta} = " + sp.latex(zeroDyn)))
display(Math(r"\eta^* = " + sp.latex(etaS) + r",\qquad \left.\frac{\partial \dot{\eta}}{\partial \eta}\right|_{\eta^*} = " + sp.latex(lam) + r" < 0"))

Die Ruhelage der Nulldynamik ist asymptotisch stabil, das System ist mit diesem Ausgang minimalphasig.
Die Steuerung folgt durch Umstellen von $\dot{y} = \dot{h}_1$ nach $u$, dafür muss $y_\mathrm{r} \in C^1$ sein.
Der Verlauf von $h_2$ wird nicht geplant und erreicht die neue Ruhelage erst nach dem Abklingen der internen Dynamik.

## Ausgang $y = h_2$: flachheitsbasierte Steuerung

Alle Systemgrößen lassen sich durch $y$, $\dot{y}$, $\ddot{y}$ ausdrücken, $y_\mathrm{r}$ muss also aus $C^2$ sein:

In [ ]:
y = sp.symbols('y', positive=True)
dy, ddy = sp.symbols(r'\dot{y} \ddot{y}', real=True)
q1 = A2 * dy + a2 * sp.sqrt(2 * g * y)
h1Flat = sp.simplify(q1 ** 2 / (2 * g * a1 ** 2))
dh1Flat = sp.diff(h1Flat, y) * dy + sp.diff(h1Flat, dy) * ddy
uFlat = A1 * dh1Flat + q1

display(Math(r"h_1 = \psi_1(y, \dot{y}) = " + sp.latex(h1Flat)))
display(Math(r"u = \psi_u(y, \dot{y}, \ddot{y}) = " + sp.latex(sp.simplify(uFlat))))

## Gegenüberstellung

$y = h_1$ |   | $y = h_2$
----------------------|----|------------------------
\begin{align*}
    u_\mathrm{d}(t) & = A_1 \dot{y}_\mathrm{r}(t) + q_1(y_\mathrm{r}(t))
\end{align*} |   | \begin{align*}
    q_{1,\mathrm{r}}(t) & = A_2 \dot{y}_\mathrm{r}(t) + q_2(y_\mathrm{r}(t)), \quad h_{1,\mathrm{r}} = \frac{q_{1,\mathrm{r}}^2}{2 g a_1^2} \\
    u_\mathrm{d}(t) & = A_1 \dot{h}_{1,\mathrm{r}}(t) + q_{1,\mathrm{r}}(t)
\end{align*}

Beide Steuerungen überführen das System von der Ruhelage zu $h_1 = h_{1,0}$ in die Ruhelage zu $h_1 = h_{1,0} + \Delta h_1$ mit der Überführungszeit $T$.

**Definition Parameter**

In [ ]:
paramsNom = st.g, st.A1, st.A2, st.a1, st.a2
tSim = np.linspace(0, 400, 801)
t0 = 20
h10 = 0.2

## Simulation

In [ ]:
output = widgets.Output()

with output:
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

fig.canvas.toolbar_visible = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False
fig.subplots_adjust(bottom=0.08, top=0.9, left=0.1, right=0.95, hspace=0.1)

ax1.set_xlim([tSim[0], tSim[-1]])
for ax in [ax1, ax2, ax3]:
    ax.grid()
ax1.set_ylabel(r"$u$ in l/s")
ax2.set_ylabel(r"$h_1$ in m")
ax3.set_ylabel(r"$h_2$ in m")
ax3.set_xlabel(r"$t$ in s")

def setLimits(ax, *data):
    data = np.concatenate([np.atleast_1d(d) for d in data])
    dMin, dMax = np.min(data), np.max(data)
    ax.set_ylim(dMin - 0.1 * np.abs(dMax - dMin) - 1e-3, dMax + 0.1 * np.abs(dMax - dMin) + 1e-3)

lineUIO, = ax1.plot([], [], 'C0', label=r'$y = h_1$')
lineUIORef, = ax1.plot([], [], 'C0:')
lineUFlat, = ax1.plot([], [], 'C1', label=r'$y = h_2$')
lineUFlatRef, = ax1.plot([], [], 'C1:')
lineH1IO, = ax2.plot([], [], 'C0')
lineH1Flat, = ax2.plot([], [], 'C1')
lineH1Ref, = ax2.plot([], [], 'k--', label=r'Referenz')
lineH2IO, = ax3.plot([], [], 'C0')
lineH2Flat, = ax3.plot([], [], 'C1')
lineH2Ref, = ax3.plot([], [], 'k--')
lineUd, = ax1.plot([], [], 'k:', label=r'$u_\mathrm{d}$ ohne Begrenzung')

handlesAx, labelsAx = [], []
for ax in [ax1, ax2, ax3]:
    h, la = ax.get_legend_handles_labels()
    handlesAx += h
    labelsAx += la

fig.legend(handlesAx, labelsAx,
           bbox_to_anchor=(0.1, 0.91, 0.85, .15), loc=3,
           ncol=4, mode="expand", borderaxespad=0., framealpha=0.5)

sliderDh = widgets.FloatSlider(value=0.1,
                               min=-0.15,
                               max=0.15,
                               step=0.01,
                               description=r"$\Delta h_1$")
sliderT = widgets.FloatSlider(value=100,
                              min=30,
                              max=300,
                              step=10,
                              description=r"$T$")
sliderDa = widgets.FloatSlider(value=0,
                               min=-20,
                               max=20,
                               step=1,
                               description=r"$\Delta a_1$ in %")
checkIO = widgets.Checkbox(value=True,
                           description=r'$y = h_1$')
checkFlat = widgets.Checkbox(value=True,
                             description=r'$y = h_2$')

def calcSystem(_):
    ffArgs = (h10, sliderDh.value, t0, sliderT.value)
    paramsSys = st.g, st.A1, st.A2, st.a1 * (1 + sliderDa.value / 100), st.a2
    x0 = [h10, stationarySys(h10, paramsNom)[1]]

    refIO = [feedForwardIO(t, *ffArgs) for t in tSim]
    refFlat = [feedForwardFlat(t, *ffArgs) for t in tSim]
    h1Ref = np.array([r[0] for r in refIO])
    uIORef = np.array([r[1] for r in refIO])
    h2Ref = np.array([r[0][1] for r in refFlat])
    uFlatRef = np.array([r[1] for r in refFlat])

    resIO = solve_ivp(lambda t, x: nonlinSys(t, x, lambda _t: feedForwardIO(_t, *ffArgs)[1], paramsSys),
                      [tSim[0], tSim[-1]], x0, t_eval=tSim, max_step=1)
    resFlat = solve_ivp(lambda t, x: nonlinSys(t, x, lambda _t: feedForwardFlat(_t, *ffArgs)[1], paramsSys),
                        [tSim[0], tSim[-1]], x0, t_eval=tSim, max_step=1)

    lines = [(checkIO.value, [(lineUIO, 1e3 * np.maximum(uIORef, 0)), (lineUIORef, 1e3 * uIORef),
                              (lineH1IO, resIO.y[0]), (lineH2IO, resIO.y[1])]),
             (checkFlat.value, [(lineUFlat, 1e3 * np.maximum(uFlatRef, 0)), (lineUFlatRef, 1e3 * uFlatRef),
                                (lineH1Flat, resFlat.y[0]), (lineH2Flat, resFlat.y[1])])]
    for active, data in lines:
        for line, d in data:
            if active:
                line.set_data(tSim, d)
            else:
                line.set_data([], [])
    lineH1Ref.set_data(tSim, h1Ref)
    lineH2Ref.set_data(tSim, h2Ref)

    setLimits(ax1, 1e3 * uIORef, 1e3 * uFlatRef)
    setLimits(ax2, h1Ref, resIO.y[0], resFlat.y[0])
    setLimits(ax3, h2Ref, resIO.y[1], resFlat.y[1])

    fig.canvas.draw()

for w in [sliderDh, sliderT, sliderDa, checkIO, checkFlat]:
    w.observe(calcSystem, names='value')

calcSystem(None)

bFeedForward = VBox([Label(value='Steuerung'), sliderDh, sliderT])
bFeedForward.layout = make_box_layout()
bOutput = VBox([Label(value='Ausgang'), checkIO, checkFlat])
bOutput.layout = make_box_layout()
bModel = VBox([Label(value='Strecke'), sliderDa])
bModel.layout = make_box_layout()
controls = HBox([bFeedForward, bOutput, bModel])

VBox([controls, output], layout=Layout(display='flex', flex_flow='column', justify_content='center', align_items='center'))